# Notebook 07D — AoU → BioMe external validation analysis

**Run date:** 20261115
**Purpose:** Post-return analysis of JJ's BioMe external validation bundle.
Produces bootstrap CIs, DeLong tests, reclassification summaries, updated
Figure 4, and Supplementary Tables S9 and S10.

**Inputs**
- `biome_external_return_bundle_20260823.zip` — JJ's return bundle
  (probabilities, performance tables, feature coverage, Table 1).
- AoU reference performance on the mutual (BioMe-compatible) feature set,
  computed in Notebook 07 Section 26 (saved to GCS).

**Primary landmark:** 1-year incident DKD.
**Primary policy:** 90% specificity on AoU test, applied as a fixed threshold
to BioMe (no re-tuning in BioMe).
**Bootstrap:** 2,000 stratified replicates (consistent with Notebooks 04/06).
**Model families (BioMe-compatible — no M5 in BioMe because no SDOH):**

| group | models | algorithms |
|---|---|---|
| `primary_biome_compatible` | M1, M2, M3, M4 (and M4a at 30d/6m) | XGB |
| `clinical_benchmark` | 4-feature clinical LR | LR |
| `algorithmic_sensitivity` | M4 | XGB / LR / RF |

**Comparisons**
1. BioMe layered gain: M1 vs M2 vs M3 vs M4 (DeLong paired).
2. BioMe algorithm choice at M4: XGB vs LR vs RF (DeLong paired).
3. BioMe clinical LR vs M4 XGB (DeLong paired).
4. Side-by-side with AoU reference on mutual features.


## 1. Setup

In [ ]:
import os, json, pathlib, warnings, datetime as dt
from collections import defaultdict
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    confusion_matrix, brier_score_loss,
)
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
warnings.filterwarnings("ignore", category=FutureWarning)

RUN_DATE = "20261115"
BOOT_N = 2000
BOOT_SEED = 42
TARGET_SPEC = 0.90

# ---- Paths ---------------------------------------------------------------
# BIOME_BUNDLE_DIR points to the extracted BioMe return bundle on the
# AoU Workbench. Override via env var if you move the folder.
BIOME_BUNDLE_DIR = pathlib.Path(os.environ.get(
    "BIOME_BUNDLE_DIR",
    "/home/jupyter/workspace/rw-migration-aou-rw-639fade3/aou_dkd/dkd_biome_results",
))

# AoU reference numbers already computed in Notebook 07 Section 26
# (BioMe-compatible models evaluated on AoU test at TARGET_SPEC).
# Export this CSV from Notebook 07 Section 26 before running Section 6 here.
AOU_REF_PATH = pathlib.Path(os.environ.get(
    "AOU_REF_PATH",
    "/home/jupyter/workspace/rw-migration-aou-rw-639fade3/aou_dkd/dkd_biome_results/"
    "aou_biome_compatible_reference_perf_20260823.csv",
))

OUT_DIR = pathlib.Path(f"./outputs_07D_{RUN_DATE}")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Outputs:", OUT_DIR.resolve())


### 1b. Helper functions

`bootstrap_test_metrics` and `delong_roc_test` are ported from Notebook 04 (cells 14 and the DeLong block). Kept here self-contained so this notebook can run without importing 04.

In [ ]:
def _metrics_at_threshold(y_true, y_prob, threshold):
    """Return a dict of classification metrics at a fixed probability threshold."""
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) else np.nan
    spec = tn / (tn + fp) if (tn + fp) else np.nan
    ppv  = tp / (tp + fp) if (tp + fp) else np.nan
    npv  = tn / (tn + fn) if (tn + fn) else np.nan
    return dict(
        auroc = roc_auc_score(y_true, y_prob),
        auprc = average_precision_score(y_true, y_prob),
        sens  = sens,
        spec  = spec,
        ppv   = ppv,
        npv   = npv,
        brier = brier_score_loss(y_true, y_prob),
        tp = tp, fp = fp, tn = tn, fn = fn,
    )

def bootstrap_test_metrics(y, p, threshold, n_boot=BOOT_N, seed=BOOT_SEED):
    """Stratified bootstrap CIs on paired (y, p) at a fixed threshold."""
    rng = np.random.default_rng(seed)
    y = np.asarray(y); p = np.asarray(p)
    idx_pos = np.where(y == 1)[0]
    idx_neg = np.where(y == 0)[0]
    if len(idx_pos) == 0 or len(idx_neg) == 0:
        raise ValueError("Need both classes for stratified bootstrap.")
    keys = ["auroc", "auprc", "sens", "spec", "ppv", "npv", "brier"]
    draws = {k: np.empty(n_boot) for k in keys}
    for b in range(n_boot):
        i_pos = rng.choice(idx_pos, size=len(idx_pos), replace=True)
        i_neg = rng.choice(idx_neg, size=len(idx_neg), replace=True)
        idx = np.concatenate([i_pos, i_neg])
        m = _metrics_at_threshold(y[idx], p[idx], threshold)
        for k in keys:
            draws[k][b] = m[k]
    out = {}
    for k in keys:
        arr = draws[k]
        out[k] = (float(np.nanpercentile(arr, 2.5)),
                  float(np.nanpercentile(arr, 97.5)))
    return out

# ----- DeLong paired AUROC test (Sun & Xu 2014 fast implementation) -------
def _compute_midrank(x):
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T
    return T2

def _fast_delong(preds_sorted_transposed, label_1_count):
    m = label_1_count
    n = preds_sorted_transposed.shape[1] - m
    positive_examples = preds_sorted_transposed[:, :m]
    negative_examples = preds_sorted_transposed[:, m:]
    k = preds_sorted_transposed.shape[0]
    tx = np.empty([k, m], dtype=float)
    ty = np.empty([k, n], dtype=float)
    tz = np.empty([k, m + n], dtype=float)
    for r in range(k):
        tx[r, :] = _compute_midrank(positive_examples[r, :])
        ty[r, :] = _compute_midrank(negative_examples[r, :])
        tz[r, :] = _compute_midrank(preds_sorted_transposed[r, :])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx[:, :]) / n
    v10 = 1.0 - (tz[:, m:] - ty[:, :]) / m
    sx = np.cov(v01)
    sy = np.cov(v10)
    delongcov = sx / m + sy / n
    return aucs, delongcov

def delong_roc_test(y_true, prob_A, prob_B):
    """Return (auc_A, auc_B, delta, z, two_sided_p) for paired ROC."""
    y_true = np.asarray(y_true)
    order = np.argsort(-y_true, kind="mergesort")
    label_1_count = int(np.sum(y_true == 1))
    preds_sorted = np.vstack([np.asarray(prob_A)[order],
                              np.asarray(prob_B)[order]])
    aucs, cov = _fast_delong(preds_sorted, label_1_count)
    l = np.array([[1, -1]])
    var = float(l @ cov @ l.T)
    se = np.sqrt(var) if var > 0 else np.nan
    delta = aucs[0] - aucs[1]
    z = delta / se if se and se > 0 else np.nan
    p = 2 * (1 - stats.norm.cdf(abs(z))) if np.isfinite(z) else np.nan
    return float(aucs[0]), float(aucs[1]), float(delta), float(z), float(p)

print("Helpers loaded.")


## 2. Load BioMe probability files

JJ's probability files follow the schema:
`<analysis_group>__<model_id>__<algorithm>__p_{raw,calibrated}`.
We default to **calibrated** probabilities for all CIs and reclassification
work; raw probabilities are kept only for the calibrated-vs-raw audit in
Section 10.


In [ ]:
PROB_FILES = {
    "30d": BIOME_BUNDLE_DIR / "biome_probabilities_30d_20260823.csv",
    "6m":  BIOME_BUNDLE_DIR / "biome_probabilities_6m_20260823.csv",
    "1y":  BIOME_BUNDLE_DIR / "biome_probabilities_1y_20260823.csv",
    "2y":  BIOME_BUNDLE_DIR / "biome_probabilities_2y_20260823.csv",
}

probs = {}
for lm, p in PROB_FILES.items():
    if not p.exists():
        print(f"  [skip] {lm}: {p} missing")
        continue
    df = pd.read_csv(p)
    probs[lm] = df
    print(f"  {lm}: n={len(df):,}  cases={int(df['dkd_binary'].sum()):,}  "
          f"prevalence={df['dkd_binary'].mean():.1%}")

# Parse probability column names into a tidy spec table for iteration.
def parse_prob_cols(df):
    rows = []
    for c in df.columns:
        if "__p_calibrated" not in c:
            continue
        stem = c.replace("__p_calibrated", "")
        parts = stem.split("__")
        if len(parts) != 3:
            continue
        group, model_id, algo = parts
        rows.append(dict(
            group=group, model_id=model_id, algorithm=algo,
            col_cal=c, col_raw=c.replace("p_calibrated", "p_raw"),
        ))
    return pd.DataFrame(rows)

prob_specs = {lm: parse_prob_cols(df) for lm, df in probs.items()}
prob_specs["1y"]


## 3. AoU reference performance (mutual feature set)

Pulled from Notebook 07 Section 26: AoU test performance using the BioMe-
compatible feature manifests, at the 90%-specificity policy, with 2,000-
replicate bootstrap CIs. These are the numbers we compare BioMe against.

If `AOU_REF_PATH` is missing, re-run Notebook 07 Section 26 to export it.


In [ ]:
if AOU_REF_PATH.exists():
    aou_ref = pd.read_csv(AOU_REF_PATH)
    print(f"AoU reference: {len(aou_ref)} rows")
    display(aou_ref.head(10))
else:
    print(f"[WARN] AoU reference not found at {AOU_REF_PATH}.")
    print("Re-export it from Notebook 07 Section 26 before running Section 6.")
    aou_ref = pd.DataFrame()


## 4. Apply AoU-derived thresholds to BioMe

Thresholds were learned on AoU calibration data and shipped with the handoff
package (`aou_biome_cal_thresholds_{RUN_DATE}.csv`). They are applied to
BioMe **without re-tuning** — this is the strict-transfer condition.


In [ ]:
thr = pd.read_csv(BIOME_BUNDLE_DIR / "aou_biome_cal_thresholds_20260823.csv")
thr90 = thr.loc[thr["target_specificity"].eq(TARGET_SPEC)].copy()
print(f"Thresholds at target_spec={TARGET_SPEC}: {len(thr90)} rows")
thr90.head(10)


In [ ]:
def lookup_threshold(group, landmark, model_id, algorithm, target_spec=TARGET_SPEC):
    mask = (
        thr["analysis_group"].eq(group)
        & thr["landmark"].eq(landmark)
        & thr["model_id"].eq(model_id)
        & thr["algorithm"].eq(algorithm)
        & thr["target_specificity"].eq(target_spec)
    )
    sub = thr.loc[mask]
    if len(sub) == 0:
        return np.nan
    return float(sub["threshold_cal"].iloc[0])

# Sanity check on 1y primary
for mid in ["M1", "M2", "M3", "M4"]:
    t = lookup_threshold("primary_biome_compatible", "1y", mid, "XGB")
    print(f"  1y {mid} XGB threshold: {t:.4f}")


## 5. BioMe bootstrap CIs

For each landmark × analysis group × model × algorithm, evaluate on BioMe
using the AoU-derived calibrated threshold, with 2,000-replicate stratified
bootstrap CIs.


In [ ]:
def evaluate_biome_row(df, spec_row, landmark):
    y = df["dkd_binary"].to_numpy()
    p = df[spec_row["col_cal"]].to_numpy()
    t = lookup_threshold(spec_row["group"], landmark,
                         spec_row["model_id"], spec_row["algorithm"])
    point = _metrics_at_threshold(y, p, t)
    ci = bootstrap_test_metrics(y, p, t)
    row = dict(
        landmark=landmark, group=spec_row["group"],
        model_id=spec_row["model_id"], algorithm=spec_row["algorithm"],
        n=int(len(y)), cases=int(y.sum()),
        threshold_cal=t,
    )
    for k, v in point.items():
        row[k] = v
    for k, (lo, hi) in ci.items():
        row[f"{k}_ci_low"] = lo
        row[f"{k}_ci_high"] = hi
    return row

biome_rows = []
for lm, df in probs.items():
    spec = prob_specs[lm]
    for _, s in spec.iterrows():
        try:
            biome_rows.append(evaluate_biome_row(df, s, lm))
        except Exception as e:
            print(f"  [skip] {lm} {s['group']} {s['model_id']} {s['algorithm']}: {e}")

biome_perf = pd.DataFrame(biome_rows)
biome_perf.to_csv(OUT_DIR / f"biome_performance_90spec_bootstrap_{RUN_DATE}.csv", index=False)
print(f"BioMe performance rows: {len(biome_perf)}")
biome_perf.loc[biome_perf["landmark"].eq("1y")].round(3)


## 6. Side-by-side AoU vs BioMe (mutual feature set)

Compare BioMe strict-transfer performance against the AoU reference computed
in Notebook 07 Section 26. The two were trained on the same mutual feature
manifest, so differences reflect cohort shift, not feature shift.


In [ ]:
if not aou_ref.empty:
    # Normalize key columns in the AoU reference so the join matches BioMe.
    aou_key = aou_ref.rename(columns={
        "model": "model_id",
    }).copy() if "model" in aou_ref.columns else aou_ref.copy()

    key_cols = ["landmark", "group", "model_id", "algorithm"]
    merge_cols = key_cols + [c for c in ["auroc", "auprc", "sens", "spec", "ppv"] if c in aou_key.columns]
    aou_sub = aou_key[merge_cols].add_suffix("_aou")
    aou_sub = aou_sub.rename(columns={f"{k}_aou": k for k in key_cols})

    compare = biome_perf.merge(aou_sub, on=key_cols, how="left",
                               suffixes=("_biome", "_aou"))
    compare["auroc_delta"] = compare["auroc"] - compare["auroc_aou"]
    compare.loc[compare["landmark"].eq("1y") & compare["group"].eq("primary_biome_compatible")].round(3)
else:
    print("[WARN] AoU reference not loaded — skipping side-by-side.")
    compare = pd.DataFrame()


## 7. BioMe DeLong paired AUROC tests

Three families of tests at each landmark (primary focus: 1y):
1. **Layered gain:** M1 vs M2, M2 vs M3, M3 vs M4 (primary group).
2. **Algorithm choice at M4:** XGB vs LR, XGB vs RF, LR vs RF
   (algorithmic_sensitivity group).
3. **Clinical benchmark:** 4-feature clinical LR vs M4 XGB.


In [ ]:
def delong_pair(df, col_A, col_B, label_A, label_B):
    y = df["dkd_binary"].to_numpy()
    pA = df[col_A].to_numpy()
    pB = df[col_B].to_numpy()
    aA, aB, d, z, p = delong_roc_test(y, pA, pB)
    return dict(
        model_A=label_A, model_B=label_B,
        auroc_A=aA, auroc_B=aB, delta=d, z=z, p_value=p,
    )

delong_rows = []

# ---- 1. Layered gain within primary BioMe-compatible models --------------
for lm, df in probs.items():
    spec = prob_specs[lm]
    prim = spec[spec["group"].eq("primary_biome_compatible")].copy()
    # Use the "step up a layer" transitions; M4a treated as M4 at 30d/6m.
    order = ["M1", "M2", "M3", "M4", "M4a"]
    prim["ord"] = prim["model_id"].map({m: i for i, m in enumerate(order)})
    prim = prim.sort_values("ord")
    cols = prim["col_cal"].tolist()
    ids  = prim["model_id"].tolist()
    for i in range(len(cols) - 1):
        r = delong_pair(df, cols[i], cols[i+1],
                        f"{lm}_{ids[i]}", f"{lm}_{ids[i+1]}")
        r["family"] = "layered_gain"
        r["landmark"] = lm
        delong_rows.append(r)

# ---- 2. Algorithm choice at 1y M4 ----------------------------------------
df1y = probs["1y"]
algo_cols = {
    "XGB": "primary_biome_compatible__M4__XGB__p_calibrated",
    "LR":  "algorithmic_sensitivity__M4__LR__p_calibrated",
    "RF":  "algorithmic_sensitivity__M4__RF__p_calibrated",
}
pairs = [("XGB", "LR"), ("XGB", "RF"), ("LR", "RF")]
for a, b in pairs:
    if algo_cols[a] in df1y.columns and algo_cols[b] in df1y.columns:
        r = delong_pair(df1y, algo_cols[a], algo_cols[b],
                        f"1y_M4_{a}", f"1y_M4_{b}")
        r["family"] = "algorithm_at_M4"
        r["landmark"] = "1y"
        delong_rows.append(r)

# ---- 3. Clinical LR vs M4 XGB at 1y --------------------------------------
clin_col = "clinical_benchmark__clinical_lr__LR__p_calibrated"
m4_col = "primary_biome_compatible__M4__XGB__p_calibrated"
if clin_col in df1y.columns and m4_col in df1y.columns:
    r = delong_pair(df1y, clin_col, m4_col, "1y_clinical_LR", "1y_M4_XGB")
    r["family"] = "clinical_vs_M4"
    r["landmark"] = "1y"
    delong_rows.append(r)

delong_df = pd.DataFrame(delong_rows)
delong_df.to_csv(OUT_DIR / f"biome_delong_tests_{RUN_DATE}.csv", index=False)
delong_df.round(4)


## 8. Reclassification across the M1→M4 cascade

For each adjacent model pair, count cases and non-cases moved into / out of
the flagged tier under the AoU-derived 90%-spec threshold. JJ already
produced a reclassification file in the bundle; we reuse it when available,
and recompute from probabilities otherwise.


In [ ]:
recls_file = BIOME_BUNDLE_DIR / "biome_external_case_reclassification_90spec_20260823.csv"
if recls_file.exists():
    recls = pd.read_csv(recls_file)
    print(f"Using JJ's reclassification file: {recls.shape}")
    display(recls.head(20))
else:
    print("[note] JJ's reclassification file not found — computing from probs.")
    rows = []
    for lm, df in probs.items():
        y = df["dkd_binary"].to_numpy()
        spec = prob_specs[lm]
        prim = spec[spec["group"].eq("primary_biome_compatible")].copy()
        order = ["M1", "M2", "M3", "M4", "M4a"]
        prim["ord"] = prim["model_id"].map({m: i for i, m in enumerate(order)})
        prim = prim.sort_values("ord").reset_index(drop=True)
        for i in range(len(prim) - 1):
            a, b = prim.iloc[i], prim.iloc[i+1]
            ta = lookup_threshold(a["group"], lm, a["model_id"], a["algorithm"])
            tb = lookup_threshold(b["group"], lm, b["model_id"], b["algorithm"])
            flag_a = (df[a["col_cal"]].to_numpy() >= ta).astype(int)
            flag_b = (df[b["col_cal"]].to_numpy() >= tb).astype(int)
            for cls, lbl in [(1, "cases"), (0, "noncases")]:
                m = (y == cls)
                rows.append(dict(
                    landmark=lm,
                    from_model=a["model_id"], to_model=b["model_id"],
                    population=lbl, n=int(m.sum()),
                    gained=int(((flag_a[m] == 0) & (flag_b[m] == 1)).sum()),
                    lost  =int(((flag_a[m] == 1) & (flag_b[m] == 0)).sum()),
                    kept_in=int(((flag_a[m] == 1) & (flag_b[m] == 1)).sum()),
                    kept_out=int(((flag_a[m] == 0) & (flag_b[m] == 0)).sum()),
                ))
    recls = pd.DataFrame(rows)
    recls["net"] = recls["gained"] - recls["lost"]
    recls["gained_pct"] = recls["gained"] / recls["n"]
    recls.to_csv(OUT_DIR / f"biome_reclassification_90spec_{RUN_DATE}.csv", index=False)
    display(recls)


## 9. Observation-density stratification (direction-reversal finding)

In AoU, AUROC improved monotonically across observation-density tertiles
(0.718 → 0.808). In BioMe, the direction reverses — this is a key support
for the observability-as-system-property thesis in the revised Discussion.

JJ shipped `biome_external_observability_tertile_summary_all_landmarks.csv`.
We re-plot it alongside the AoU version for the paper.


In [ ]:
tert_file = BIOME_BUNDLE_DIR / "biome_external_observability_tertile_summary_all_landmarks_20260823.csv"
if tert_file.exists():
    biome_tert = pd.read_csv(tert_file)
    print(f"BioMe observability tertiles: {biome_tert.shape}")
    display(biome_tert.loc[biome_tert['landmark'].eq('1y')] if 'landmark' in biome_tert.columns else biome_tert.head())
else:
    print(f"[WARN] {tert_file} missing — skipping.")
    biome_tert = pd.DataFrame()


## 10. Calibrated vs raw probability audit

Compare point AUROC on BioMe using raw vs isotonic-calibrated outputs. This
is a sanity check that calibration did not materially shift discrimination
(it affects threshold placement, not ranking).


In [ ]:
audit_rows = []
for lm, df in probs.items():
    y = df["dkd_binary"].to_numpy()
    spec = prob_specs[lm]
    for _, s in spec.iterrows():
        if s["col_raw"] not in df.columns:
            continue
        a_raw = roc_auc_score(y, df[s["col_raw"]].to_numpy())
        a_cal = roc_auc_score(y, df[s["col_cal"]].to_numpy())
        audit_rows.append(dict(
            landmark=lm, group=s["group"],
            model_id=s["model_id"], algorithm=s["algorithm"],
            auroc_raw=a_raw, auroc_cal=a_cal, delta=a_cal - a_raw,
        ))
audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(OUT_DIR / f"biome_raw_vs_cal_audit_{RUN_DATE}.csv", index=False)
audit_df.loc[audit_df['landmark'].eq('1y')].round(4)


## 11. Figure 4 — External validation (updated)

Panels:
- **a.** AoU vs BioMe AUROC bars at 1y across M1–M4, mutual feature set,
  with 95% CIs.
- **b.** AoU vs BioMe sensitivity at the fixed 90%-specificity threshold.
- **c.** Observation-density tertile AUROC (AoU: monotonic rise; BioMe:
  direction reverses).
- **d.** Reclassification waterfall across M1→M2→M3→M4 at 1y.


In [ ]:
def _fmt_ci(row, metric):
    return f"{row[metric]:.3f} [{row[f'{metric}_ci_low']:.3f}–{row[f'{metric}_ci_high']:.3f}]"

# Placeholder figure skeleton — fill in after confirming AoU reference load.
fig, axes = plt.subplots(2, 2, figsize=(11, 8.5))
ax = axes.ravel()

# Panel a: AUROC bars at 1y, M1–M4
sub = biome_perf.loc[
    biome_perf["landmark"].eq("1y") &
    biome_perf["group"].eq("primary_biome_compatible") &
    biome_perf["algorithm"].eq("XGB") &
    biome_perf["model_id"].isin(["M1", "M2", "M3", "M4"])
].sort_values("model_id")
x = np.arange(len(sub))
w = 0.38
ax[0].bar(x - w/2, sub["auroc"], width=w, label="BioMe",
          yerr=[sub["auroc"] - sub["auroc_ci_low"],
                sub["auroc_ci_high"] - sub["auroc"]],
          capsize=3, color="#4477AA")
if not aou_ref.empty and "auroc" in aou_ref.columns:
    aou_sub = aou_ref.loc[
        aou_ref["landmark"].eq("1y") &
        aou_ref["group"].eq("primary_biome_compatible") &
        aou_ref["algorithm"].eq("XGB") &
        aou_ref["model_id"].isin(["M1", "M2", "M3", "M4"])
    ].sort_values("model_id")
    if len(aou_sub) == len(sub):
        ax[0].bar(x + w/2, aou_sub["auroc"].values, width=w, label="AoU",
                  color="#CC6677")
ax[0].set_xticks(x)
ax[0].set_xticklabels(sub["model_id"])
ax[0].set_ylabel("AUROC")
ax[0].set_title("a. AUROC at 1y (mutual features)")
ax[0].set_ylim(0.5, 0.9)
ax[0].legend(frameon=False)

# Panel b: Sensitivity at fixed 90% specificity
ax[1].bar(x - w/2, sub["sens"], width=w,
          yerr=[sub["sens"] - sub["sens_ci_low"],
                sub["sens_ci_high"] - sub["sens"]],
          capsize=3, color="#4477AA", label="BioMe")
ax[1].set_xticks(x)
ax[1].set_xticklabels(sub["model_id"])
ax[1].set_ylabel("Sensitivity @ 90% specificity")
ax[1].set_title("b. Sensitivity at fixed threshold")
ax[1].set_ylim(0, 0.7)

# Panel c: Observability tertile AUROC (placeholder — fill from biome_tert)
ax[2].set_title("c. AUROC by observation-density tertile (TODO)")
ax[2].text(0.5, 0.5, "Fill from AoU vs BioMe tertile summary",
           ha="center", va="center", transform=ax[2].transAxes)

# Panel d: Reclassification waterfall (placeholder — fill from recls)
ax[3].set_title("d. Reclassification across cascade (TODO)")
ax[3].text(0.5, 0.5, "Fill from recls dataframe", ha="center", va="center",
           transform=ax[3].transAxes)

for a in ax:
    a.spines["top"].set_visible(False)
    a.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(OUT_DIR / f"figure_4_external_validation_{RUN_DATE}.pdf",
            bbox_inches="tight")
plt.savefig(OUT_DIR / f"figure_4_external_validation_{RUN_DATE}.png",
            dpi=300, bbox_inches="tight")
plt.show()


## 12. Supplementary Table S9 — BioMe detailed metrics

Per-landmark × group × model row with point estimate and 95% CI for each
metric, plus the applied threshold and the DeLong p-value vs the previous
layer (where applicable).


In [ ]:
def _ci_cell(point, lo, hi, dp=3):
    return f"{point:.{dp}f} [{lo:.{dp}f}, {hi:.{dp}f}]"

s9 = biome_perf.copy()
for m in ["auroc", "auprc", "sens", "spec", "ppv", "npv", "brier"]:
    s9[f"{m}_disp"] = [_ci_cell(r[m], r[f"{m}_ci_low"], r[f"{m}_ci_high"])
                       for _, r in s9.iterrows()]

# Join DeLong p-values for the "vs prev layer" comparisons.
dl_layered = delong_df.loc[delong_df["family"].eq("layered_gain")].copy()
dl_layered["model_id"] = dl_layered["model_B"].str.split("_").str[-1]
dl_layered["landmark"] = dl_layered["landmark"]
s9 = s9.merge(
    dl_layered[["landmark", "model_id", "p_value"]].rename(
        columns={"p_value": "p_vs_prev_layer"}),
    on=["landmark", "model_id"], how="left",
)

cols = [
    "landmark", "group", "model_id", "algorithm",
    "n", "cases", "threshold_cal",
    "auroc_disp", "auprc_disp", "sens_disp", "spec_disp",
    "ppv_disp", "npv_disp", "brier_disp", "p_vs_prev_layer",
]
s9_out = s9[cols].rename(columns={
    "auroc_disp": "AUROC [95% CI]",
    "auprc_disp": "AUPRC [95% CI]",
    "sens_disp":  "Sensitivity [95% CI]",
    "spec_disp":  "Specificity [95% CI]",
    "ppv_disp":   "PPV [95% CI]",
    "npv_disp":   "NPV [95% CI]",
    "brier_disp": "Brier [95% CI]",
    "p_vs_prev_layer": "DeLong p vs prev layer",
})
s9_out.to_csv(OUT_DIR / f"supp_table_S9_biome_detailed_metrics_{RUN_DATE}.csv",
              index=False)
s9_out.loc[s9_out['landmark'].eq('1y')]


## 13. Supplementary Table S10 — BioMe feature coverage

JJ supplied feature coverage files already. We assemble a two-part S10:
- **S10a:** coverage by domain (counts + overall coverage %).
- **S10b:** coverage by individual feature (presence rate in BioMe cohort,
  by outcome where available).


In [ ]:
cov_dom = pd.read_csv(BIOME_BUNDLE_DIR / "biome_external_feature_coverage_by_domain_20260823.csv")
cov_feat = pd.read_csv(BIOME_BUNDLE_DIR / "biome_external_feature_coverage_by_feature_20260823.csv")
cov_dom.to_csv(OUT_DIR / f"supp_table_S10a_feature_coverage_by_domain_{RUN_DATE}.csv", index=False)
cov_feat.to_csv(OUT_DIR / f"supp_table_S10b_feature_coverage_by_feature_{RUN_DATE}.csv", index=False)
display(cov_dom.head(15))
display(cov_feat.head(10))


## 14. Export manifest and run summary

In [ ]:
manifest = {
    "run_date": RUN_DATE,
    "biome_bundle": str(BIOME_BUNDLE_DIR),
    "target_specificity": TARGET_SPEC,
    "bootstrap_n": BOOT_N,
    "bootstrap_seed": BOOT_SEED,
    "landmarks": list(probs.keys()),
    "outputs": sorted(str(p.name) for p in OUT_DIR.glob("*.csv")) +
               sorted(str(p.name) for p in OUT_DIR.glob("*.pdf")) +
               sorted(str(p.name) for p in OUT_DIR.glob("*.png")),
}
with open(OUT_DIR / f"run_manifest_07D_{RUN_DATE}.json", "w") as f:
    json.dump(manifest, f, indent=2, default=str)
print(json.dumps(manifest, indent=2, default=str))
